# Capability・Recommendation実践

このNotebookで確認すること：問題要件、4種類の判定、候補の列挙、必要引数、実際の候補生成。適合は性能の優位性を意味しません。

[Theory](../../docs/theory/13_model_selection.md) / [Guide](../../docs/guide/13_model_selection.md) / [利用方法](../../docs/capability_workflow.md)

ProblemSpec → Recommendation → Data → Model → Fit → Posterior → Acquisition → Optimization → Candidate → Evaluation

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from robotorchan.models import SingleTaskGP, create_model, get_model_class, list_models
from robotorchan.acquisition import (
    create_acquisition, get_acquisition_class, list_acquisitions,
)
from robotorchan.optim import (
    create_optimizer, create_search_strategy, list_optimizers, list_search_strategies,
)
from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace

torch.manual_seed(17)
dtype = torch.double
from collections import Counter
from robotorchan.workflow import (
    ProblemSpec, evaluate_components, check_compatibility, recommend,
)
from botorch.acquisition.objective import GenericMCObjective

## 1. 条件による候補集合の違い

回帰と分類、出力数と目的数、観測制約と候補制約は別々に指定します。高次元・頑健・multi-fidelity・選好などは追加要件です。判定結果の件数は、学習前に候補集合の規模を把握するために利用できます。

In [ ]:
specs = {
    "scalar": ProblemSpec(purpose="bayesian_optimization"),
    "mixed": ProblemSpec(purpose="bayesian_optimization", input_type="mixed", q=2),
    "multiobjective": ProblemSpec(
        purpose="bayesian_optimization", output_type="multi", objective_type="multi", q=2
    ),
    "outcome_constraint": ProblemSpec(
        purpose="bayesian_optimization", output_type="multi", constrained=True, q=2
    ),
    "candidate_constraint": ProblemSpec(
        purpose="bayesian_optimization", candidate_constrained=True, q=2
    ),
    "binary": ProblemSpec(purpose="active_learning", observation_kind="binary", q=2),
    "multiclass": ProblemSpec(purpose="active_learning", observation_kind="multiclass", q=2),
    "ordinal": ProblemSpec(purpose="active_learning", observation_kind="ordinal", q=2),
    "multi_fidelity": ProblemSpec(purpose="bayesian_optimization", multi_fidelity=True),
    "multi_task": ProblemSpec(purpose="bayesian_optimization", task_type="multitask"),
    "high_dimensional": ProblemSpec(purpose="bayesian_optimization", high_dimensional=True),
    "robust": ProblemSpec(purpose="bayesian_optimization", robust=True),
    "preference": ProblemSpec(purpose="bayesian_optimization", preference=True),
}
for name, spec in specs.items():
    inventory = evaluate_components(spec)
    print(name)
    for group in ("models", "acquisitions", "optimizers", "search_strategies"):
        print(" ", group, dict(Counter(item.status.value for item in getattr(inventory, group))))

## 2. 不適合理由・条件付き判定・自動列挙

q=2にanalytic EIを指定する例、分類ALに回帰モデルを指定する例を比較します。autoは候補の列挙であって自動学習ではありません。

In [ ]:
for spec, model_name, acq_name in [
    (specs["mixed"], "MixedSingleTaskGP", "LogExpectedImprovement"),
    (specs["binary"], "SingleTaskGP", "qPredictiveEntropy"),
    (specs["scalar"], "SingleTaskGP", "qLogExpectedImprovement"),
]:
    decision = check_compatibility(
        spec, model=model_name, acquisition=acq_name, optimizer="sobol"
    )
    print(decision.status, decision.reasons, decision.conditions)

alternatives = recommend(
    specs["scalar"], model="SingleTaskGP",
    acquisition="qLogExpectedImprovement", optimizer="auto",
)
for item in alternatives:
    print(item.optimizer_name, item.compatibility.status, item.required_arguments())

## 3. 単目的・候補制約・観測制約の実行

ここからは選択された構成を実際に学習します。候補制約は入力xに課し、観測制約は予測出力g(x) <= 0に課します。後者は確率的な制約であり、真の実験結果が必ず実行可能とは限りません。

In [ ]:
X = torch.linspace(0.1, 0.9, 9, dtype=dtype).unsqueeze(-1)
Y = -((X - 0.65) ** 2)
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
grid = torch.linspace(0, 1, 101, dtype=dtype).unsqueeze(-1)
for key in ("scalar", "candidate_constraint", "outcome_constraint"):
    spec = specs[key]
    (choice,) = recommend(
        spec, model="SingleTaskGP", acquisition="qLogExpectedImprovement", optimizer="sobol"
    )
    print(key, choice.required_arguments(), choice.compatibility.conditions)
    observations = torch.cat([Y, X - 0.8], dim=-1) if key == "outcome_constraint" else Y
    model = create_model(choice.model_name, X, observations)
    model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
    with torch.no_grad():
        mean = model.posterior(grid).mean
    acq_kwargs = {"best_f": Y.max()}
    if key == "outcome_constraint":
        acq_kwargs["objective"] = GenericMCObjective(lambda samples, X=None: samples[..., 0])
        acq_kwargs["constraints"] = [lambda samples: samples[..., 1]]
    acq = create_acquisition(choice.acquisition_name, model=model, **acq_kwargs)
    optimize = create_optimizer(choice.optimizer_name, raw_samples=32, seed=17)
    options = {}
    if key == "candidate_constraint":
        options["constraints"] = CandidateConstraints(
            inequality_constraints=((torch.tensor([0]), torch.tensor([1.], dtype=dtype), 0.2),),
        )
    candidate, score = optimize(acq, bounds, q=spec.q, **options)
    assert candidate.shape == (spec.q, 1)
    if key == "candidate_constraint":
        assert torch.all(candidate >= 0.2 - 1e-8)
    print(candidate, score, -((candidate - 0.65) ** 2))
    plt.figure(figsize=(7, 3))
    plt.plot(grid.squeeze(), mean[:, 0], label="Posterior objective")
    plt.scatter(X.squeeze(), Y.squeeze(), label="Observations")
    plt.scatter(candidate.squeeze(), -((candidate.squeeze() - 0.65) ** 2), label="Candidates")
    plt.title(key)
    plt.legend()
    plt.show()

## 4. Mixed・多目的の構成選択と候補生成

各ケースの必要引数を表示し、選択された名前をFactoryへ渡します。

In [ ]:
(choice,) = recommend(
    specs["mixed"], model="MixedSingleTaskGP",
    acquisition="qLogExpectedImprovement", optimizer="sobol",
)
print(choice.required_arguments())
Xm = torch.cat([X, (torch.arange(len(X)) % 2).to(dtype).unsqueeze(-1)], dim=-1)
Ym = Y + 0.1 * Xm[:, 1:]
bm = torch.tensor([[0., 0.], [1., 1.]], dtype=dtype)
model = create_model(choice.model_name, Xm, Ym, cat_dims=[1])
model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
acq = create_acquisition(choice.acquisition_name, model=model, best_f=Ym.max())
optimize = create_optimizer(choice.optimizer_name, raw_samples=32, seed=17)
candidate, score = optimize(
    acq, bm, q=2, variable_space=MixedVariableSpace(bounds=bm, integer_dims=(1,))
)
assert candidate.shape == (2, 2)
assert torch.equal(candidate[:, 1], candidate[:, 1].round())
print(candidate, score)

In [ ]:
(choice,) = recommend(
    specs["multiobjective"], model="SingleTaskGP",
    acquisition="qLogExpectedHypervolumeImprovement", optimizer="sobol",
)
print(choice.required_arguments())
Y2 = torch.cat([Y, -((X - 0.25) ** 2)], dim=-1)
model = create_model(choice.model_name, X, Y2)
model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
ref = Y2.min(dim=0).values - 0.1
acq = create_acquisition(
    choice.acquisition_name, model=model, ref_point=ref.tolist(),
    partitioning=FastNondominatedPartitioning(ref_point=ref, Y=Y2),
    sampler=SobolQMCNormalSampler(torch.Size([16]), seed=17),
)
optimize = create_optimizer(choice.optimizer_name, raw_samples=32, seed=17)
candidate, score = optimize(acq, bounds, q=2)
assert candidate.shape == (2, 1)
new_y = torch.cat([-((candidate - 0.65) ** 2), -((candidate - 0.25) ** 2)], dim=-1)
print(candidate, score, new_y)
plt.scatter(Y2[:, 0], Y2[:, 1], label="Observations")
plt.scatter(new_y[:, 0], new_y[:, 1], label="Candidates")
plt.legend()
plt.show()

## 5. 分類AL：Search Strategyを選択する

二値・多クラス・順序分類を別の観測形式として指定します。実データでは検証データを用いて学習回数や予測確率の較正を評価してください。

In [ ]:
for key, name, k in [
    ("binary", "BinarySingleTaskGPClassifier", 2),
    ("multiclass", "MulticlassSingleTaskGPClassifier", 3),
    ("ordinal", "OrdinalSingleTaskGPClassifier", 3),
]:
    (choice,) = recommend(
        specs[key], model=name, acquisition="qPredictiveEntropy",
        search_strategy="SobolSearchStrategy",
    )
    print(choice.required_arguments())
    labels = (X.squeeze(-1) * k).long()
    kwargs = {"num_classes": k} if k > 2 else {}
    model = create_model(choice.model_name, X, labels, **kwargs)
    model.fit(epochs=2)
    with torch.no_grad():
        probabilities = model.predict_proba(grid)
    acq = create_acquisition(choice.acquisition_name, model=model)
    strategy = create_search_strategy(choice.search_strategy_name, bounds, num_samples=32, seed=17)
    candidate = strategy.optimize(acq, q=2).candidates
    assert candidate.shape == (2, 1)
    print(key, candidate)
    plt.figure(figsize=(7, 3))
    plt.plot(grid.squeeze(), probabilities)
    for x in candidate.flatten():
        plt.axvline(x.item(), linestyle="--")
    plt.title(key)
    plt.show()

## 6. モデル固有の構造とRecommendation

変分推論、忠実度、タスク、選好比較では学習データと必要引数が異なります。構成選択の結果を確認し、モデル固有の引数を明示して候補を生成します。

In [ ]:
optimize = create_optimizer("sobol", raw_samples=32, seed=17)
native_opt = create_optimizer("botorch", num_restarts=2, raw_samples=16)

### 6.1 Variational GP

Exact GPと異なり、誘導点を用いた変分推論を行います。fitの引数も異なります。Factoryはこれらを共通化せず、元のコンストラクタと学習契約を維持します。

In [ ]:
spec = specs["scalar"]
choices = recommend(spec, model="SingleTaskVariationalGP", acquisition="qLogExpectedImprovement", optimizer="sobol")
if not choices:
    decision = check_compatibility(
        spec, model="SingleTaskVariationalGP", acquisition="qLogExpectedImprovement", optimizer="sobol"
    )
    print(decision.status, decision.reasons, decision.conditions)
else:
    choice = choices[0]
    print(choice.required_arguments(), choice.compatibility.conditions)
    vgp = create_model(choice.model_name, X, Y, inducing_points=5)
    vgp.fit(epochs=10, lr=0.03, batch_size=9)
    vgp.eval()
    vgp.likelihood.eval()
    with torch.no_grad():
        vp = vgp.posterior(grid)
    va = create_acquisition(choice.acquisition_name, model=vgp, best_f=Y.max())
    cv, vv = optimize(va, bounds, q=2)
    assert cv.shape == (2, 1)
    print(cv, vv, -((cv - 0.65) ** 2))
    plt.plot(grid.squeeze(), vp.mean.squeeze(), label="Variational GP")
    plt.scatter(X.squeeze(), Y.squeeze())
    plt.scatter(cv.squeeze(), -((cv.squeeze() - 0.65) ** 2))
    plt.legend()
    plt.show()

### 6.2 Multi-Fidelity GP：高忠実度での候補提案

第2列は忠実度です。低忠実度には系統的な偏りを加えます。ここでは最終忠実度1に固定した候補提案を示します。評価コストを考慮して忠実度自体を選ぶMFKGとは区別します。

In [ ]:
spec = specs["multi_fidelity"]
choices = recommend(spec, model="SingleTaskMultiFidelityGP", acquisition="PosteriorMean", optimizer="botorch")
if not choices:
    decision = check_compatibility(
        spec, model="SingleTaskMultiFidelityGP", acquisition="PosteriorMean", optimizer="botorch"
    )
    print(decision.status, decision.reasons, decision.conditions)
else:
    choice = choices[0]
    print(choice.required_arguments(), choice.compatibility.conditions)
    fidelity = (torch.arange(len(X)) % 2).to(dtype).unsqueeze(-1) * 0.5 + 0.5
    Xf = torch.cat([X, fidelity], dim=-1)
    Yf = Y + 0.2 * (1 - fidelity)
    mf = create_model(choice.model_name, Xf, Yf, data_fidelities=[1])
    mf.fit(optimizer_kwargs={"options": {"maxiter": 10}})
    af = create_acquisition(choice.acquisition_name, model=mf)
    bf = torch.tensor([[0., 0.5], [1., 1.]], dtype=dtype)
    native_opt = create_optimizer("botorch", num_restarts=2, raw_samples=16)
    cf, vf = native_opt(af, bf, q=1, fixed_features={1: 1.0})
    assert cf.shape == (1, 2)
    assert cf[0, 1].item() == 1.0
    print(cf, vf, -((cf[:, :1] - 0.65) ** 2))
    with torch.no_grad():
        for f in (0.5, 1.0):
            pg = mf.posterior(torch.cat([grid, torch.full_like(grid, f)], dim=-1))
            plt.plot(grid.squeeze(), pg.mean.squeeze(), label=f"Fidelity {f}")
    plt.legend()
    plt.show()

### 6.3 Multi-Task GP：対象タスクを指定する

task_featureは学習入力のタスク列、output_tasksは予測対象です。対象タスクの候補はタスク列を含まない物理変数空間で生成します。

In [ ]:
spec = specs["multi_task"]
choices = recommend(spec, model="MultiTaskGP", acquisition="qLogExpectedImprovement", optimizer="botorch")
if not choices:
    decision = check_compatibility(
        spec, model="MultiTaskGP", acquisition="qLogExpectedImprovement", optimizer="botorch"
    )
    print(decision.status, decision.reasons, decision.conditions)
else:
    choice = choices[0]
    print(choice.required_arguments(), choice.compatibility.conditions)
    Xt = torch.cat([
        torch.cat([X, torch.zeros_like(X)], dim=-1),
        torch.cat([X, torch.ones_like(X)], dim=-1),
    ])
    Yt = torch.cat([Y + 0.1, -((X - 0.4) ** 2)])
    mt = create_model(choice.model_name, Xt, Yt, task_feature=1, output_tasks=[1])
    mt.fit(optimizer_kwargs={"options": {"maxiter": 10}})
    at = create_acquisition(choice.acquisition_name, model=mt, best_f=Yt[len(X):].max())
    ct, vt = native_opt(at, bounds, q=1)
    assert ct.shape == (1, 1)
    print(ct, vt, -((ct - 0.4) ** 2))
    with torch.no_grad():
        pt = mt.posterior(grid)
    plt.plot(grid.squeeze(), pt.mean.squeeze(), label="Target task")
    plt.scatter(X.squeeze(), Yt[len(X):].squeeze())
    plt.scatter(ct.squeeze(), -((ct.squeeze() - 0.4) ** 2))
    plt.legend()
    plt.show()

### 6.4 Preference BO：比較データから候補ペアを提案

観測値は実数ではなく勝者・敗者のインデックスです。AnalyticExpectedUtilityOfBestOptionは2候補のうち好まれる方の潜在効用を評価します。

In [ ]:
spec = ProblemSpec(purpose="bayesian_optimization", preference=True, q=2)
choices = recommend(spec, model="PairwiseGP", acquisition="AnalyticExpectedUtilityOfBestOption", optimizer="botorch")
if not choices:
    decision = check_compatibility(
        spec, model="PairwiseGP", acquisition="AnalyticExpectedUtilityOfBestOption", optimizer="botorch"
    )
    print(decision.status, decision.reasons, decision.conditions)
else:
    choice = choices[0]
    print(choice.required_arguments(), choice.compatibility.conditions)
    datapoints = X.clone()
    pairs = [(i, i + 1) if Y[i] >= Y[i + 1] else (i + 1, i) for i in range(len(X) - 1)]
    comparisons = torch.tensor(pairs, dtype=torch.long)
    pref = create_model(choice.model_name, datapoints=datapoints, comparisons=comparisons)
    pref.fit(optimizer_kwargs={"options": {"maxiter": 10}})
    ap = create_acquisition(choice.acquisition_name, pref_model=pref)
    cp, vp = native_opt(ap, bounds, q=2)
    assert cp.shape == (2, 1)
    print(cp, vp, "preferred index:", (-((cp - 0.65) ** 2)).argmax().item())
    with torch.no_grad():
        pp = pref.posterior(grid)
    plt.plot(grid.squeeze(), pp.mean.squeeze(), label="Latent utility")
    for x in cp.flatten():
        plt.axvline(x.item(), linestyle="--")
    plt.legend()
    plt.show()

## 7. 次元削減・非定常・非GPモデル

PCAとPLSは低次元表現を利用しますが、候補は元の入力空間で生成します。非定常GPは入力位置に依存する変動を表現します。Random Forestは木の集合に基づく予測を利用するため、勾配を必要としない探索を選びます。

In [ ]:
Xh = torch.rand(20, 6, dtype=dtype)
Yh = -((Xh[:, :2] - 0.65) ** 2).sum(dim=-1, keepdim=True)
bh = torch.stack([torch.zeros(6, dtype=dtype), torch.ones(6, dtype=dtype)])
for name in ("PCAGP", "PLSGP", "RandomProjectionGP"):
    spec = ProblemSpec(purpose="bayesian_optimization", high_dimensional=True, q=2)
    (choice,) = recommend(
        spec, model=name, acquisition="qLogExpectedImprovement", optimizer="sobol"
    )
    print(choice.required_arguments())
    surrogate = create_model(choice.model_name, Xh, Yh, n_components=2)
    acquisition_name = choice.acquisition_name
    optimizer_name = choice.optimizer_name
    surrogate.fit()
    with torch.no_grad():
        posterior = surrogate.posterior(Xh)
    acquisition = create_acquisition(acquisition_name, model=surrogate, best_f=Yh.max())
    solver = create_optimizer(optimizer_name, raw_samples=32, seed=17)
    point, value = solver(acquisition, bh, q=2)
    assert point.shape == (2, 6)
    print(name, posterior.mean.shape, point, value)
    print("Objective:", -((point[:, :2] - 0.65) ** 2).sum(dim=-1))

In [ ]:
Xn = torch.linspace(0, 1, 16, dtype=dtype).unsqueeze(-1)
Yn = torch.where(Xn < 0.5, torch.sin(2 * torch.pi * Xn), torch.sin(8 * torch.pi * Xn))
for name, kwargs, acquisition_name in [
    ("NonstationarySingleTaskGP", {}, "qLogExpectedImprovement"),
    ("RandomForestSurrogate", {"n_estimators": 32, "random_state": 17}, "qExpectedImprovement"),
]:
    spec = ProblemSpec(purpose="bayesian_optimization", robust=name.startswith("Nonstationary"))
    choices = recommend(spec, model=name, acquisition=acquisition_name, optimizer="sobol")
    if not choices:
        decision = check_compatibility(
            spec, model=name, acquisition=acquisition_name, optimizer="sobol"
        )
        print(name, decision.status, decision.reasons)
        continue
    choice = choices[0]
    surrogate = create_model(choice.model_name, Xn, Yn, **kwargs)
    acquisition_name = choice.acquisition_name
    optimizer_name = choice.optimizer_name
    surrogate.fit()
    acquisition = create_acquisition(acquisition_name, model=surrogate, best_f=Yn.max())
    solver = create_optimizer(optimizer_name, raw_samples=32, seed=17)
    point, value = solver(acquisition, bounds, q=1)
    assert point.shape == (1, 1)
    with torch.no_grad():
        mean = surrogate.posterior(grid).mean.squeeze()
    plt.figure(figsize=(7, 3))
    plt.plot(grid.squeeze(), mean, label=name)
    plt.scatter(Xn.squeeze(), Yn.squeeze())
    plt.axvline(point.item(), linestyle="--")
    plt.legend()
    plt.show()
    print(point, value)

## 8. Optimizerと探索方式の比較

同じ学習済みGPと獲得関数を使用し、ランダム探索、Sobol探索、BoTorchの勾配ベース最適化を比較します。探索予算と初期化の違いがあるため、単一試行の獲得値だけで手法の優劣は判断できません。

In [ ]:
comparison_model = create_model("SingleTaskGP", X, Y)
comparison_model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
comparison_acq = create_acquisition(
    "LogExpectedImprovement", model=comparison_model, best_f=Y.max()
)
for name in ("random", "sobol", "botorch"):
    (choice,) = recommend(
        specs["scalar"], model="SingleTaskGP",
        acquisition="LogExpectedImprovement", optimizer=name,
    )
    name = choice.optimizer_name
    solver = create_optimizer(name, raw_samples=32, num_restarts=2)
    point, value = solver(comparison_acq, bounds, q=1)
    assert point.shape == (1, 1)
    print(name, point.item(), value.item())
for name in ("RandomSearchStrategy", "SobolSearchStrategy"):
    (choice,) = recommend(
        specs["scalar"], model="SingleTaskGP",
        acquisition="LogExpectedImprovement", search_strategy=name,
    )
    name = choice.search_strategy_name
    search = create_search_strategy(name, bounds, num_samples=32, seed=17)
    result = search.optimize(comparison_acq, q=1)
    assert result.candidates.shape == (1, 1)
    print(name, result.candidates, result.acquisition_value)

## 9. 実務での適合判定と関連資料

静的な適合判定は構成選択の出発点です。学習後の予測精度、posteriorの形状、獲得関数の評価、候補点の実行可能性を実データで確認してください。

モデル固有の学習方法と候補生成については、次の実践例を参照できます。

- [Multi-Fidelity GP](03_multi_fidelity_gp.ipynb)
- [Multi-Task GP](04_multitask_gp.ipynb)
- [高次元BO](54_high_dimensional_bo_benchmark.ipynb)
- [Robust GP](13_robust_gp.ipynb)
- [Preference BO](24_pairwise_gp.ipynb)
- [個別Factoryによる候補提案](62_unified_factories.ipynb)